# Defining Hooks

This lesson introduces the **`queries`** project used for the rest of the hooks section.

Hooks intercept and control tool calls before/after they run, giving you fine-grained control over what Claude can and can't do.


## Building a hook — 4 steps

1. **PreToolUse or PostToolUse** — Pre can *prevent* a tool call; Post runs after it already happened.
2. **Which tools to watch** — the `matcher` specifies exactly which tool(s) trigger your hook.
3. **Write a command that receives the tool call** — it gets **JSON about the proposed tool call on stdin**.
4. **Provide feedback (if needed)** — the command's **exit code** tells Claude whether to allow or block.

## Available tools to watch

Claude Code's built-ins include **Read, Write, Edit, MultiEdit, Bash, Glob, Grep, WebFetch, WebSearch, Task** (and more). The exact set can **change when you add MCP servers** (each adds `mcp__<server>__<tool>` tools).

> Tip from the lesson: just **ask Claude for the current list** of available tools — most reliable, since it reflects your MCP setup.

## Tool-call data structure (stdin)

When your hook runs, Claude Code pipes JSON like this to it:

```json
{
  "session_id": "2d6a1e4d-6...",
  "transcript_path": "/Users/sg/...",
  "hook_event_name": "PreToolUse",
  "tool_name": "Read",
  "tool_input": {
    "file_path": "/code/queries/.env"
  }
}
```

Your command reads this from **standard input**, parses it, and decides based on `tool_name` + `tool_input`.

## Exit codes & control flow

- **Exit `0`** — all good, allow the tool call.
- **Exit `2`** — **block** the tool call (PreToolUse only).

On exit `2` in a PreToolUse hook, whatever you write to **stderr is sent to Claude as feedback**, explaining why it was blocked.

**Example use case — block reading `.env`:** since both **Read** and **Grep** can access file contents, watch *both* and block if the target path is a restricted file (like `.env`).

## The project: `queries`

The hooks section uses a TypeScript project. Two copies (same pattern as the MCP section):

- **[`./queries/`](./queries)** — **starter**. `hooks/read_hook.js` has the `TODO` you'll implement next lesson. Its editable hook files are `git skip-worktree`, so your local edits stay out of git and the starter stays pristine.
- **[`./queries-complete/`](./queries-complete)** — **answer key** (the `.env`-blocking hook implemented, plus `pre-log.json`/`post-log.json` samples).

Key files:
- `hooks/read_hook.js` — reads the tool-call JSON from stdin; you add the `.env` check here (next lesson).
- `hooks/query_hook.js`, `hooks/tsc.js` — other hook scripts used in later lessons.
- `.claude/settings.example.json` — sample hook config (copy to `.claude/settings.json` to activate). It wires `query_hook.js` on Write/Edit, prettier + `tsc.js` on PostToolUse, and `jq`-based logging.
- `.env` — a **deliberate dummy** (`SECRET_API_KEY="SUPER SECRET API KEY"`) that exists purely so the read-blocking demo has something to protect. Force-committed here despite the repo's usual `.env` ignore, because it's fake and the demo needs it.

> Config note: `settings.example.json` uses `jq`, `npx prettier`, and Node — so running the full example needs **Node + jq** installed. The read-blocking hook itself is just Node.

Next: **Implementing a hook** — write the `.env` check in `read_hook.js` and wire it up.